# 02 — Separate rest and movement windows

**MATLAB equivalent:** `metadata2.m`

For each subject, this notebook:

1. Finds repetition boundaries using `rerepetition`.
2. Keeps repetitions 1–6.
3. Finds contiguous rest/movement segments using `restimulus`.
4. Generates overlapping windows inside each segment.
5. Stores rest and movement windows separately, including repetition IDs and original sample indices.

**Parameters:** 200 Hz sampling, 40 samples/window (200 ms), 10 samples/step (50 ms).

In [ ]:
from pathlib import Path
import numpy as np
import joblib

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / "data" / "NinaPro_DB5"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_SUBJECTS = 10
N_CHANNELS = 16
FS = 200
WINDOW_SAMPLES = 40
STEP_SAMPLES = 10
CLASS_ORDER = np.arange(13)  # 0 = rest, 1–12 = movements

all_E1 = joblib.load(OUTPUT_DIR / "all_E1.joblib")
assert len(all_E1) == N_SUBJECTS, "Expected recordings for 10 subjects."

## Window-generation function

Windows are never allowed to cross a repetition or movement/rest segment boundary. Segments shorter than one complete window are skipped, matching the MATLAB implementation.

In [ ]:
def empty_segment_store():
    return {
        "EMG": [],
        "RepetitionLabel": [],
        "StartSample": [],
        "EndSample": [],
    }

def generate_subject_windows(record):
    emg = np.asarray(record["emg"], dtype=np.float64)
    movement = np.asarray(record["restimulus"]).reshape(-1)
    repetition = np.asarray(record["rerepetition"]).reshape(-1)

    if len(emg) != len(movement) or len(emg) != len(repetition):
        raise ValueError("EMG, movement and repetition lengths do not match.")
    if emg.shape[1] != N_CHANNELS:
        raise ValueError(f"Expected {N_CHANNELS} EMG channels.")

    rest = empty_segment_store()
    move = {
        "EMG": [],
        "MovementLabel": [],
        "RepetitionLabel": [],
        "StartSample": [],
        "EndSample": [],
    }

    # MATLAB: [1; find(diff(repetition) ~= 0)+1; length(repetition)+1]
    rep_boundaries = np.r_[0, np.where(np.diff(repetition) != 0)[0] + 1, len(repetition)]

    for k in range(len(rep_boundaries) - 1):
        rep_start = int(rep_boundaries[k])       # zero-based, inclusive
        rep_end = int(rep_boundaries[k + 1])     # zero-based, exclusive
        rep_id_value = repetition[rep_start]

        if rep_id_value not in range(1, 7):
            continue
        rep_id = int(rep_id_value)

        rep_movement = movement[rep_start:rep_end]

        seg_boundaries = np.r_[
            0, np.where(np.diff(rep_movement) != 0)[0] + 1, len(rep_movement)
        ]

        for j in range(len(seg_boundaries) - 1):
            seg_start = int(seg_boundaries[j])
            seg_end = int(seg_boundaries[j + 1])
            label_value = rep_movement[seg_start]

            if label_value < 0 or label_value > 12:
                continue
            if not float(label_value).is_integer():
                continue
            label = int(label_value)

            segment = emg[rep_start + seg_start : rep_start + seg_end]
            n_samples = len(segment)

            if n_samples < WINDOW_SAMPLES:
                continue

            n_windows = (n_samples - WINDOW_SAMPLES) // STEP_SAMPLES + 1

            for w in range(n_windows):
                local_start = w * STEP_SAMPLES
                local_end = local_start + WINDOW_SAMPLES
                window = segment[local_start:local_end]

                # Zero-based original sample indices, inclusive at both ends.
                global_start = rep_start + seg_start + local_start
                global_end = rep_start + seg_start + local_end - 1

                if label == 0:
                    rest["EMG"].append(window)
                    rest["RepetitionLabel"].append(rep_id)
                    rest["StartSample"].append(global_start)
                    rest["EndSample"].append(global_end)
                else:
                    move["EMG"].append(window)
                    move["MovementLabel"].append(label)
                    move["RepetitionLabel"].append(rep_id)
                    move["StartSample"].append(global_start)
                    move["EndSample"].append(global_end)

    # Convert metadata lists to arrays; EMG remains a list of window arrays.
    for store in (rest, move):
        for key in store:
            if key != "EMG":
                store[key] = np.asarray(store[key], dtype=np.int64)

    return {"Rest": rest, "Movement": move}

## Process all subjects and save

In [ ]:
WindowedDB5 = {}

for s, record in enumerate(all_E1, start=1):
    subject_name = f"S{s}"
    WindowedDB5[subject_name] = generate_subject_windows(record)

    n_rest = len(WindowedDB5[subject_name]["Rest"]["EMG"])
    n_move = len(WindowedDB5[subject_name]["Movement"]["EMG"])
    print(f"{subject_name}: Rest windows={n_rest:,}, Movement windows={n_move:,}")

joblib.dump(WindowedDB5, OUTPUT_DIR / "WindowedDB5.joblib", compress=3)
print("\\nWindow generation completed and saved.")